# SHAP Lag Variable Investigation
This notebook contains the 3 plots discussed to identify why the model's SHAP value for the lag variable (`FLOW_lag_24h`) increases or decreases.

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

shap.initjs()

### 1. Load Data and Prepare Features
We replicate the data preparation pipeline to get the exact `X_test` features used by the baseline model.

In [ ]:
print("Loading data...")
df = pl.read_csv("cleaned_master.csv", schema_overrides={"SITEREF": pl.String, "SH": pl.String})
df = df.with_columns(pl.col("DATETIME").str.to_datetime())

df = df.with_columns([
    pl.col("DATETIME").dt.hour().alias("HOUR"),
    pl.col("DATETIME").dt.weekday().alias("DAY_OF_WEEK"),
    pl.col("DATETIME").dt.month().alias("MONTH"),
    pl.col("DATETIME").dt.year().alias("YEAR")
])

df = df.with_columns(
    pl.when(pl.col('UNIQUE_ID').str.ends_with('_2'))
    .then(pl.lit(1)).otherwise(pl.lit(0)).cast(pl.Int8).alias('IS_DIRECTION_2')
)
df = df.with_columns(pl.col('DATETIME').dt.truncate('1h').alias('DATETIME_HOUR'))

df_hourly = df.group_by(['DATETIME_HOUR', 'UNIQUE_ID', 'SITEREF']).agg([
    pl.col('FLOW').sum().alias('FLOW'),
    pl.col('TEMP').mean(), pl.col('RH').mean(), pl.col('WDSP').mean(),
    pl.col('DEWP').mean(), pl.col('VISIB').mean(), pl.col('GUST').mean(),
    pl.col('IS_EXTREME').max(), pl.col('IS_HOLIDAY').max(), pl.col('IS_PANDEMIC').max(),
    pl.col('IS_DIRECTION_2').max(), pl.col('EXTREME_HAZARD').first(), pl.col('EXTREME_ID').first(),
    pl.col('HOLIDAY_NAME').first(), pl.col('HOUR').first(), pl.col('DAY_OF_WEEK').first(),
    pl.col('MONTH').first(), pl.col('YEAR').first(), pl.col('REGION').first(),
    pl.col('SH').first(), pl.col('LANE').first(),
]).sort(['UNIQUE_ID', 'DATETIME_HOUR'])

df_hourly = df_hourly.with_columns(
    pl.when(pl.col('DAY_OF_WEEK') >= 6).then(1).otherwise(0).alias('IS_WEEKEND')
)

df_hourly = df_hourly.with_columns(pl.col('REGION').str.replace(r'^\d+ - ', '').alias('REGION_NAME')).with_columns([
    (pl.col('REGION_NAME') == 'Auckland').cast(pl.Int8).alias('IS_AUCKLAND'),
    (pl.col('REGION_NAME') == 'Wellington').cast(pl.Int8).alias('IS_WELLINGTON'),
    (pl.col('REGION_NAME') == 'Canterbury').cast(pl.Int8).alias('IS_CANTERBURY'),
])

unique_holidays = df.filter((pl.col('IS_HOLIDAY') == 1) & (pl.col('HOLIDAY_NAME').is_not_null()) & (pl.col('HOLIDAY_NAME') != 'None'))['HOLIDAY_NAME'].unique().to_list()
unique_extremes = df.filter((pl.col('IS_EXTREME') == 1) & (pl.col('EXTREME_HAZARD') != 'None'))['EXTREME_HAZARD'].unique().to_list()

one_hot_exprs = []
for holiday in unique_holidays:
    clean_name = holiday.replace(" ", "_").replace("'", "")
    one_hot_exprs.append(pl.when((pl.col('IS_HOLIDAY') == 1) & (pl.col('HOLIDAY_NAME') == holiday)).then(1).otherwise(0).alias(f'HOLIDAY_{clean_name}'))
for extreme_id in unique_extremes:
    one_hot_exprs.append(pl.when((pl.col('IS_EXTREME') == 1) & (pl.col('EXTREME_HAZARD') == extreme_id)).then(1).otherwise(0).alias(f'EVENT_{extreme_id}'))
df_hourly = df_hourly.with_columns(one_hot_exprs)

df_hourly = df_hourly.with_columns([
    (2 * np.pi * pl.col("HOUR") / 24).sin().alias("HOUR_SIN"),
    (2 * np.pi * pl.col("HOUR") / 24).cos().alias("HOUR_COS"),
    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).sin().alias("DAY_SIN"),
    (2 * np.pi * pl.col("DAY_OF_WEEK") / 7).cos().alias("DAY_COS"),
    (2 * np.pi * pl.col("MONTH") / 12).sin().alias("MONTH_SIN"),
    (2 * np.pi * pl.col("MONTH") / 12).cos().alias("MONTH_COS")
])

df_hourly = df_hourly.with_columns([
    pl.col('FLOW').shift(1).over('UNIQUE_ID').alias('FLOW_lag_1h'),
    pl.col('FLOW').shift(24).over('UNIQUE_ID').alias('FLOW_lag_24h'),
    pl.col('FLOW').shift(168).over('UNIQUE_ID').alias('FLOW_lag_168h'),
    pl.col('FLOW').shift(1).rolling_mean(window_size=4).over('UNIQUE_ID').alias('FLOW_roll_mean_4h'),
])

WEATHER_FEATURES  = ['TEMP', 'RH', 'WDSP', 'DEWP', 'VISIB', 'GUST']
NRC_FEATURES = ['IS_PANDEMIC', 'IS_WEEKEND', 'HOLIDAY_ANZAC_Day', 'HOLIDAY_Christmas_Day', 'HOLIDAY_New_Years_Day', 'HOLIDAY_Boxing_Day', 'HOLIDAY_Waitangi_Day', 'HOLIDAY_Day_after_New_Years_Day', 'HOLIDAY_Queens_Birthday', 'HOLIDAY_Labour_Day', 'HOLIDAY_Easter_Monday', 'HOLIDAY_Good_Friday', 'EVENT_Multi Hazard, Snow / Ice', 'EVENT_High Wind / Gust', 'EVENT_Multi Hazard', 'EVENT_Snow / Ice', 'EVENT_Flooding']
SPECIFIC_REGIONS  = ['IS_AUCKLAND', 'IS_WELLINGTON', 'IS_CANTERBURY']
TIME_FEATURES     = ['HOUR_SIN', 'HOUR_COS', 'DAY_SIN', 'DAY_COS', 'MONTH_SIN', 'MONTH_COS', 'YEAR']
TRAFFIC_FEATURES  = ['IS_DIRECTION_2']
LAG_FEATURES      = ['FLOW_lag_24h']

ALL_FEATURES = TRAFFIC_FEATURES + NRC_FEATURES + SPECIFIC_REGIONS + TIME_FEATURES + WEATHER_FEATURES + LAG_FEATURES

for feature in ALL_FEATURES:
    if feature not in df_hourly.columns:
        df_hourly = df_hourly.with_columns(pl.lit(0).alias(feature))

df_hourly = df_hourly.drop_nulls(subset=ALL_FEATURES)
test = df_hourly.filter(pl.col('YEAR') == 2021).to_pandas().sort_values('DATETIME_HOUR')
X_test = test[ALL_FEATURES]

print(f"Data loaded successfully! X_test shape: {X_test.shape}")

### 2. Load Model and Compute Standard SHAP Values
We load `baseline_xgb.joblib` and compute the SHAP values. To keep computation fast for plotting, we'll sample 5000 records.

In [ ]:
print("Loading baseline_xgb.joblib...")
model = joblib.load('baseline_xgb.joblib')

print("Computing SHAP values... (this might take a minute)")
explainer = shap.TreeExplainer(model)
X_subset = X_test.sample(n=5000, random_state=42)
shap_values = explainer.shap_values(X_subset)
print("SHAP values computed!")

### Plot 1: SHAP Dependence Plot (Marginal Effect)
This plot shows the pure relationship between the actual value of `FLOW_lag_24h` (x-axis) and its SHAP value (y-axis).
Notice the shape of the curve: Does it increase linearly, or does it plateau at high volumes?

In [ ]:
shap.dependence_plot("FLOW_lag_24h", shap_values, X_subset, interaction_index=None)

### Plot 2: SHAP Dependence Plot with Interaction Index
Now we plot the same curve, but we color the dots based on another feature (`IS_WEEKEND`).

Notice if the weekend dots (red) cluster lower or higher than the weekday dots (blue) for the same lag volume. If they do, the model is discounting the lag variable during weekends.

In [ ]:
shap.dependence_plot("FLOW_lag_24h", shap_values, X_subset, interaction_index="IS_WEEKEND")

### Plot 3: SHAP Interaction Values (Advanced)
To mathematically isolate the interaction effect from the main effect, we compute **SHAP Interaction Values**. This creates a matrix of interactions.

This plot specifically isolates the pure interaction effect between the lag variable and the weekend feature. If the Y-value is negative when `IS_WEEKEND`=1, it proves the weekend condition explicitly suppresses the lag's importance.

In [ ]:
print("Computing SHAP interaction values... (this takes longer, sampling 1000 records)")
X_interact = X_test.sample(n=1000, random_state=42)
shap_interaction_values = explainer.shap_interaction_values(X_interact)

shap.dependence_plot(
    ("FLOW_lag_24h", "IS_WEEKEND"), 
    shap_interaction_values, 
    X_interact
)